# Synthetic Q&A Generation for Evaluation

This notebook uses the Gemini API to generate synthetic Question and Answer pairs from the source documents (`.jsonl`). These pairs will serve as our "ground truth" for evaluating the RAG system.

In [ ]:
import os
import json
import random
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

load_dotenv()
DATA_DIR = "../data/final"
OUTPUT_FILE = "../data/evaluation_dataset.json"

## 1. Load Sample Texts
We randomly sample a few paragraphs from our processed data to act as context for Q&A generation.

In [ ]:
samples = []
files_to_sample = [f for f in os.listdir(DATA_DIR) if f.endswith('.jsonl')]

for filename in files_to_sample:
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if lines:
            # Take 5 random lines from each file
            sampled_lines = random.sample(lines, min(5, len(lines)))
            for line in sampled_lines:
                data = json.loads(line)
                text = data.get("text", "") or data.get("content", "")
                if len(text.split()) > 50: # Only keep substantial paragraphs
                    samples.append({"source": filename, "text": text})

# Shuffle and limit to 50 samples for cost/time efficiency
random.shuffle(samples)
samples = samples[:50]
print(f"Selected {len(samples)} contexts for Q&A generation.")

## 2. Generate Q&A using Gemini API
We use `langchain_google_genai` to ask the LLM to generate a question that can be answered *only* using the provided text.

In [ ]:
class QAItem(BaseModel):
    question: str = Field(description="A specific question based on the text in Indonesian.")
    answer: str = Field(description="The ground truth answer derived solely from the text in Indonesian.")

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)
parser = JsonOutputParser(pydantic_object=QAItem)

prompt = PromptTemplate(
    template="""Anda adalah ahli Teologi Katolik yang bertugas membuat soal evaluasi.
Berdasarkan teks berikut, buatlah SATU pasang Pertanyaan dan Jawaban yang spesifik.
Jawaban HARUS dapat ditemukan secara eksplisit di dalam teks.
Gunakan Bahasa Indonesia yang baku.

Teks Konteks:
{context}

Format Output:
{format_instructions}""",
    input_variables=["context"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser

In [ ]:
evaluation_data = []

print("Generating...")
for i, sample in enumerate(samples):
    try:
        result = chain.invoke({"context": sample["text"]})
        evaluation_data.append({
            "question": result["question"],
            "ground_truth": result["answer"],
            "context": sample["text"],
            "source": sample["source"]
        })
        if (i+1) % 10 == 0: print(f"Done {i+1}/{len(samples)}")
    except Exception as e:
        print(f"Error on sample {i}: {e}")

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(evaluation_data, f, ensure_ascii=False, indent=2)
    
print(f"Saved {len(evaluation_data)} Q&A pairs to {OUTPUT_FILE}")